In [1]:
import os
import sys
sys.path.append('..')
import torch
import librosa
from IPython import display as disp
from tqdm import tqdm
from src.modules import ResynthesisWithQuantAndTVTimbreModule, SpeakerModule

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "2" # Set the GPU device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sample_rate = 16000

In [3]:
speaker_encoder = SpeakerModule(device="cuda")
resynthesizer = ResynthesisWithQuantAndTVTimbreModule.load_from_checkpoint(
    "../ckpts/resynthesis_quant_and_tvtimbre_epoch=17-val_loss=1.8843.ckpt",
    encoder_path=None,
    map_location="cuda"
)

resynthesizer = resynthesizer.eval()

/data/waris/installations/darkstream/lib/python3.10/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)
/data/waris/installations/darkstream/lib/python3.10/site-packages/speechbrain/utils/checkpoints.py:200: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they ar

In [4]:
@torch.no_grad()
def inference_non_streaming(source_audio_path, tgt_embed, sample_rate=16000):
    src_audio, _ = librosa.load(source_audio_path, sr=sample_rate)
    src_audio = torch.from_numpy(src_audio).unsqueeze(0).unsqueeze(0).to(device)
    
    resyth_audio = resynthesizer(src_audio, tgt_embed)
    resyth_audio = resyth_audio.squeeze()
    resyth_audio = resyth_audio.cpu().numpy()
    
    return resyth_audio

In [5]:
@torch.no_grad()
def inference_streaming(source_audio_path, tgt_embed, sample_rate=16000):
    '''
    Streaming inference function for resynthesis with quantized content encoder and TV-Timbre decoder.
    Processes the input audio in chunks with lookahead to simulate streaming behavior.
    Args:
        source_audio_path (str): Path to the source audio file.
        tgt_embed (torch.Tensor): Target speaker embedding tensor.
        sample_rate (int): Sample rate for loading the audio.
    Returns:
    resyth_audio (np.ndarray): The resynthesized audio as a numpy array.
    '''
    src_audio, _ = librosa.load(source_audio_path, sr=sample_rate)
    src_audio = torch.from_numpy(src_audio).unsqueeze(0).unsqueeze(0).to(device)
    
    original_length = src_audio.shape[-1]

    # Set streaming parameters
    chunk_size = 3  # in number of frames, each frame is 20ms, so chunk_size=3 means 60ms chunks. Can be changed. min is 1.
    frame_samples = 320 # 20 ms at 16kHz, frame rate = 50 fps, don't change this value.
    lookahead_size = 4  # Don't change this value.

    # The model advances by 'stride' frames each step.
    stride = chunk_size * frame_samples

    # We always need lookahead frames available for the last step.
    lookahead_samples = lookahead_size * frame_samples

    # Number of stride-sized chunks needed to cover original_length
    # (ceil division)
    num_strides = (original_length + stride - 1) // stride

    # Total length required so that:
    # - audio length is a multiple of stride
    # - plus lookahead tail for the final window
    required_length = num_strides * stride + lookahead_samples

    pad_length = max(0, required_length - original_length)

    if pad_length > 0:
        pad = src_audio.new_zeros((*src_audio.shape[:-1], pad_length))
        audio_padded = torch.cat([src_audio, pad], dim=-1)
    else:
        audio_padded = src_audio


    all_audios = []
    all_z_embeds = []
    embed_stack = torch.empty((1, 512, 0)).to(device)
    embed_pointer = 0
    with resynthesizer.encoder.net.streaming(batch_size=1): # set encoder to streaming mode to enable streaming states (cache)
        with resynthesizer.decoder.streaming(batch_size=1): # set decoder to streaming mode to enable streaming states (cache)
            for offset in tqdm(range(0, original_length, stride + lookahead_samples)):  # in steps of frame_size, so chunk size is 20ms
                frame = audio_padded[:, :, offset:offset + stride + lookahead_samples]  # add lookahead for quant model

                enc_state = resynthesizer.encoder.net._streaming_state
                with resynthesizer.encoder.net._context_for_encoder_decoder:
                    emb = resynthesizer.encoder.net.encoder(frame)

                embed_stack = torch.cat((embed_stack, emb), dim=-1)
                if embed_stack.shape[-1] - embed_pointer < lookahead_size + chunk_size:
                    continue

                while embed_stack.shape[-1] - embed_pointer >= lookahead_size + chunk_size:
                    emb = embed_stack[:, :, embed_pointer:embed_pointer + lookahead_size + chunk_size]
                    embed_pointer += chunk_size

                    assert enc_state.graphed_tr_enc is not None
                    (emb,) = enc_state.graphed_tr_enc(emb, lookahead_size)
                    z_embed, *_ = resynthesizer.encoder.net.quantizer(emb)
                    if isinstance(z_embed, tuple):
                        z_embed = z_embed[0]
                    if z_embed.shape[-1] == 0:
                        continue

                    z_embed = z_embed[:, :, :chunk_size]
                    resynth_audio_chunk, _ = resynthesizer.decoder(z_embed, tgt_embed)

                    all_audios.append(resynth_audio_chunk)
                    all_z_embeds.append(z_embed)
    all_audios = torch.cat(all_audios, dim=-1)
    all_z_embeds = torch.cat(all_z_embeds, dim=-1)

    resyth_audio = all_audios.squeeze().cpu().numpy()
    return resyth_audio

In [7]:
wav_file = "../sample_wav/ASI/arctic_a0001.wav"
target_file = "../sample_wav/SLT/arctic_a0001.wav"

# Load target speaker embedding
audio, _ = librosa.load(target_file, sr=sample_rate)
audio = torch.from_numpy(audio).unsqueeze(0).unsqueeze(0).to(device)
tgt_embed = speaker_encoder(audio.squeeze(0))

In [11]:
# Run Non-Streaming Inference
resyth_audio = inference_non_streaming(wav_file, tgt_embed, sample_rate=16000)
disp.Audio(resyth_audio, rate=16000)

In [10]:
# Run Streaming Inference
resyth_audio_streaming = inference_streaming(wav_file, tgt_embed, sample_rate=16000)
disp.Audio(resyth_audio_streaming, rate=16000)

100%|██████████| 23/23 [00:01<00:00, 20.22it/s]
